# 収益性ファクターのサイズ・バリュー調整：米国でも効果はあるか

**問題意識**：日本株では、収益性（ROE等）でそのまま銘柄を選ぶより、**まずサイズで分け、さらにバリューで調整する** と
収益性のパフォーマンスがはっきり表れる傾向がある。同じ効果が **米国（MSCI World 米国構成銘柄）** でも見られるかを確認する。

**分析の流れ**（ユニバースは米国内のみ。日本は最後に参考として同じ手順で比較）

| ステップ | 収益性の並べ方 | 目的 |
|---|---|---|
| 0. 無条件 | 全銘柄で収益性を並べる | 基準 |
| 1. サイズで条件付け | サイズ階層ごとに収益性を並べ、階層間で平均 | サイズの影響を除く |
| 2. サイズ→調整変数の2段ソート | サイズ階層 × 調整変数階層の中で収益性を並べ、平均 | バリュー等の影響を除く（条件付きソート） |
| 2b. 合成スコア | サイズ階層内で z(収益性)+z(調整変数) | 「割安な高収益」を直接選ぶ |
| 2c. 残差 | サイズ階層内で収益性を調整変数に回帰した残差 | 調整変数で説明できない収益性 |

- **調整変数**：B/P（バリュー）を中心に、E/P、低投資（総資産成長率・投資/総資産の逆）、株主還元（総還元利回り・純還元利回り）も比較
- **ウェイト**：等ウェイト（EW）と時価総額加重（VW）の両方
- **改善の出どころ**：サイズ階層別・バリュー階層別、ロング側/ショート側、期間別に分解
- 各比較は **同じ銘柄サンプル** の上で行う（サンプルの違いによる見かけの改善を避けるため）
- **先読みバイアス対策**：財務特性は `CHAR_LAG_MONTHS` か月前の値（実績が判明した後）を使い、時価総額は組成月の値を使う
- **サバイバーシップ対策**：組成時点で翌月リターンの有無を問わず並べる。指数除外・上場廃止などでリターンが得られない銘柄は、
  その資金を **組成時のウェイト（EWなら等ウェイト、VWなら当初の時価総額ウェイト）** で同じポートフォリオの残りの銘柄に再配分する
- **追加分析（9章）**：収益性は長期の投資ホライズンを想定するため、**12か月保有（バイ・アンド・ホールド）** でも評価する

上から順に実行すれば完結します（pickle は使っていません）。

## 0. 設定

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401  日本語フォント設定

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

COUNTRY_COLORS = {"US": "#1f4e9c", "JP": "#c0392b"}
COUNTRY_LABELS = {"US": "米国", "JP": "日本"}
WEIGHT_LABELS = {"EW": "等ウェイト", "VW": "時価総額加重"}

In [ ]:
# ===== パラメータ =====
START_DATE = 200001
END_DATE = 999999

PROF_VAR = "roe_actual"        # 収益性指標（roe_actual / roa_actual / profitability_factor など）
RET_COL = "return_1m_forward_local_inc_div_pct"   # 将来1か月リターン（現地通貨・配当込み, %）
CHAR_LAG_MONTHS = 3            # 財務特性（収益性・調整変数）を何か月前の値で使うか。実績の判明を待つためのラグ
                               # （時価総額・セクターは組成月の値。2章の診断で更新タイミングを確認して調整）
LAG_SENSITIVITY = [0, 3, 6]    # ラグの感応度分析
HOLD_MONTHS = 12               # 追加分析：保有期間（月）

N_SIZE = 3                     # サイズ階層数（1=小型 … N=大型。MSCI World内での相対的な大小）
N_ADJ = 3                      # 調整変数の階層数
N_PROF = 3                     # 収益性の階層数（ロングショート = 最上位 − 最下位）
N_PROF_UNCOND = 5              # 無条件の分位ポートフォリオ（単調性の確認用）
MIN_STOCKS = 5                 # ポートフォリオの最小銘柄数（未満の月は欠損）

EXCLUDE_SECTOR_CODES = ["40"]  # 金融を除外。含める場合は []
WINSOR_LOWER, WINSOR_UPPER = 0.01, 0.99   # 特性値の月次ウィンザー化（合成・残差の計算用）
NW_LAGS = 6                    # Newey-West t値のラグ（月次リターン）
ROLL_MONTHS = 36               # ローリング指標の窓

# 調整変数： 名前 → (列名, 向き)。向き +1 は「高いほど良い」、-1 は「低いほど良い」
ADJUSTERS = {
    "B/P": ("book_to_price_actual", +1),
    "E/P": ("earnings_to_price_actual", +1),
    "低投資（総資産成長率）": ("asset_growth_actual", -1),
    "低投資（投資/総資産）": ("investment_to_assets_actual", -1),
    "総還元利回り": ("gross_payout_yield", +1),
    "純還元利回り": ("net_payout_yield", +1),
}
MAIN_ADJ = "B/P"               # 詳細分析の対象とする調整変数

PERIODS = {"2000-2009": (2000, 2009), "2010-2019": (2010, 2019), "2020-": (2020, 2100)}

USE_DUMMY_DATA = False         # True で架空データによる動作確認（結果に意味はない）

## 1. データ取得

`fetch_data(QUERY, "irddb")` は社内DB接続用の関数です。普段お使いの import 文を下のセルに記入してください。

In [ ]:
# 社内DB接続関数の読み込み（環境に合わせて書き換えてください）
# from xxxxx import fetch_data

if not USE_DUMMY_DATA and "fetch_data" not in globals():
    raise NameError("fetch_data が定義されていません。上の import 文を環境に合わせて記入してください。")

In [ ]:
def get_profitability_data(country="US", start_date=200001, end_date=999999):
    country = country.strip().upper()

    if country not in {"US", "JP"}:
        raise ValueError("countryは'US'または'JP'を指定してください。")

    start_date = int(start_date)
    end_date = int(end_date)

    QUERY = f"""
    DECLARE @StartDate INT = {start_date};
    DECLARE @EndDate   INT = {end_date};

    SELECT
        -- 識別・管理
        F.[DATE]    AS [year_month],
        F.[GID]     AS [Global_ID],
        F.[BID]     AS [Barra_ID],
        F.[SEDOL]   AS [SEDOL],
        F.[UPDDATE] AS [update_date],

        -- MSCI銘柄情報
        NULLIF(M.[NAME], '-999999998')
            AS [msci_name],
        M.[CTRY]
            AS [msci_country],
        NULLIF(M.[GICS], '00000000')
            AS [gics_code],

        -- 企業規模
        NULLIF(F.[TMV_USD], -999999998)
            AS [company_market_cap_usd],
        NULLIF(F.[MV_USD], -999999998)
            AS [security_market_cap_usd],

        -- 実績収益性
        NULLIF(F.[ROE_ACT], -999999998)
            AS [roe_actual],
        NULLIF(F.[ROA_ACT], -999999998)
            AS [roa_actual],

        -- 予想収益性
        NULLIF(F.[ROE_EST], -999999998)
            AS [roe_estimate],
        NULLIF(F.[ROA_EST], -999999998)
            AS [roa_estimate],

        -- 収益性ファクター
        NULLIF(F.[PROF], -999999998)
            AS [profitability_factor],

        -- 収益性の源泉
        NULLIF(F.[GM_ACT], -999999998)
            AS [gross_margin_actual],
        NULLIF(F.[OIS_ACT], -999999998)
            AS [operating_margin_actual],
        NULLIF(F.[NIS_ACT], -999999998)
            AS [net_margin_actual],
        NULLIF(F.[TA_TO], -999999998)
            AS [asset_turnover_actual],

        -- 財務構成・利益の質
        NULLIF(F.[EQR], -999999998)
            AS [equity_ratio],
        NULLIF(F.[TD_TA], -999999998)
            AS [debt_to_assets],
        NULLIF(F.[OCF_TA], -999999998)
            AS [operating_cf_to_assets],

        -- バリュエーション
        NULLIF(F.[BP_ACT], -999999998)
            AS [book_to_price_actual],
        NULLIF(F.[EP_ACT], -999999998)
            AS [earnings_to_price_actual],

        -- 成長・投資
        NULLIF(F.[SALES_GRW], -999999998)
            AS [sales_growth_actual],
        NULLIF(F.[TA_GRW], -999999998)
            AS [asset_growth_actual],
        NULLIF(F.[IA], -999999998)
            AS [investment_to_assets_actual],

        -- 株主還元：配当
        NULLIF(F.[DP_ACT], -999999998)
            AS [dividend_yield_actual],
        NULLIF(F.[DP_EST], -999999998)
            AS [dividend_yield_estimate],
        NULLIF(F.[DOE], -999999998)
            AS [dividend_on_equity],

        -- 株主還元：自社株買い・総還元・純還元
        NULLIF(F.[BUYBACK], -999999998)
            AS [buyback_yield],
        NULLIF(F.[GPY], -999999998)
            AS [gross_payout_yield],
        NULLIF(F.[NPY], -999999998)
            AS [net_payout_yield]

    FROM [GLOBAL].[quants].[FACTOR] AS F

    INNER JOIN [GLOBAL].[quants].[MSCI] AS M
        ON M.[DATE] = F.[DATE]
        AND M.[GID]  = F.[GID]

    WHERE F.[DATE] BETWEEN @StartDate AND @EndDate
        AND F.[BID] <> 'XXXXXXX'
        AND M.[F_IDX_DM] = 1
        AND M.[CTRY] = '{country}'

    ORDER BY
        F.[DATE],
        F.[GID];
    """

    df = fetch_data(QUERY, "irddb")

    # 文字列の前後の空白を除去（pandas 2.1未満は applymap）
    strip = lambda x: x.strip() if isinstance(x, str) else x
    df = df.map(strip) if hasattr(df, "map") else df.applymap(strip)

    return df

In [ ]:
def get_return_data(country="US", start_date=200001, end_date=999999):
    country = country.strip().upper()

    if country not in {"US", "JP"}:
        raise ValueError("countryは'US'または'JP'を指定してください。")

    start_date = int(start_date)
    end_date = int(end_date)

    if start_date > end_date:
        raise ValueError("start_dateはend_date以下にしてください。")

    QUERY = f"""
    DECLARE @StartDate INT = {start_date};
    DECLARE @EndDate   INT = {end_date};

    SELECT
        -- 識別・管理
        R.[DATE]    AS [year_month],
        R.[GID]     AS [Global_ID],
        R.[BID]     AS [Barra_ID],
        R.[SEDOL]   AS [SEDOL],
        R.[UPDDATE] AS [mkt_update_date],

        -- 過去1か月リターン：ローカル通貨建て（単位：%）
        NULLIF(R.[RTNP], -999999998)
            AS [return_1m_past_local_ex_div_pct],
        NULLIF(R.[DRTNP], -999999998)
            AS [return_1m_past_local_inc_div_pct],

        -- 将来1か月リターン：ローカル通貨建て（単位：%）
        NULLIF(R.[RTNF], -999999998)
            AS [return_1m_forward_local_ex_div_pct],
        NULLIF(R.[DRTNF], -999999998)
            AS [return_1m_forward_local_inc_div_pct],

        -- 過去1か月リターン：米ドル建て（単位：%）
        NULLIF(R.[RTNP_USD], -999999998)
            AS [return_1m_past_usd_ex_div_pct],
        NULLIF(R.[DRTNP_USD], -999999998)
            AS [return_1m_past_usd_inc_div_pct],

        -- 将来1か月リターン：米ドル建て（単位：%）
        NULLIF(R.[RTNF_USD], -999999998)
            AS [return_1m_forward_usd_ex_div_pct],
        NULLIF(R.[DRTNF_USD], -999999998)
            AS [return_1m_forward_usd_inc_div_pct],

        -- 過去12か月リターン：ローカル通貨建て（単位：%）
        NULLIF(R.[RTNP12], -999999998)
            AS [return_12m_past_local_ex_div_pct],
        NULLIF(R.[DRTNP12], -999999998)
            AS [return_12m_past_local_inc_div_pct],

        -- 将来12か月リターン：ローカル通貨建て（単位：%）
        NULLIF(R.[RTNF12], -999999998)
            AS [return_12m_forward_local_ex_div_pct],
        NULLIF(R.[DRTNF12], -999999998)
            AS [return_12m_forward_local_inc_div_pct],

        -- 12か月リターンの計算に使用された有効月数
        R.[AVAILMP] AS [return_12m_past_valid_months],
        R.[AVAILMF] AS [return_12m_forward_valid_months]

    FROM [GLOBAL].[quants].[MKT] AS R

    WHERE R.[DATE] BETWEEN @StartDate AND @EndDate
        AND R.[BID] <> 'XXXXXXX'
        AND EXISTS (
            SELECT 1
            FROM [GLOBAL].[quants].[MSCI] AS M
            WHERE M.[DATE] = R.[DATE]
                AND M.[GID]  = R.[GID]
                AND M.[F_IDX_DM] = 1
                AND M.[CTRY] = '{country}'
        )

    ORDER BY
        R.[DATE],
        R.[GID];
    """

    df = fetch_data(QUERY, "irddb")

    # 文字列の前後の空白を除去（pandas 2.1未満は applymap）
    strip = lambda x: x.strip() if isinstance(x, str) else x
    df = df.map(strip) if hasattr(df, "map") else df.applymap(strip)

    return df

In [ ]:
def make_dummy_data(country, start_date=200001, end_date=202412, n_firms=None, seed=0):
    """動作確認用の架空データ（USE_DUMMY_DATA=True のときのみ使用）。分析上の意味はない。"""
    rng = np.random.default_rng(seed + (0 if country == "US" else 1))
    n_firms = n_firms or (400 if country == "US" else 200)
    dates = pd.period_range(str(max(start_date, 200001))[:4] + "-" + str(max(start_date, 200001))[4:],
                            str(min(end_date, 202412))[:4] + "-" + str(min(end_date, 202412))[4:], freq="M")
    sectors = ["10102010", "15101010", "20101010", "25101010", "30101010", "35101010",
               "40101010", "45101010", "50101010", "55101010", "60101010"]
    firm_sector = rng.choice(sectors, n_firms)
    base_nm = rng.normal(0.08 if country == "US" else 0.05, 0.05, n_firms)
    base_to = rng.lognormal(np.log(0.7 if country == "US" else 0.8), 0.4, n_firms)
    base_eq = rng.uniform(0.2, 0.6 if country == "US" else 0.7, n_firms)
    rows = []
    nm, to, eq = base_nm.copy(), base_to.copy(), base_eq.copy()
    mcap = rng.lognormal(np.log(2e10 if country == "US" else 8e9), 1.0, n_firms)
    for d in dates:
        nm = 0.97 * nm + 0.03 * base_nm + rng.normal(0, 0.005, n_firms)
        to = to * np.exp(rng.normal(0, 0.02, n_firms))
        eq = np.clip(eq + rng.normal(0, 0.005, n_firms), 0.05, 0.95)
        mcap = mcap * np.exp(rng.normal(0.005, 0.07, n_firms))
        roa = nm * to
        roe = roa / eq
        df = pd.DataFrame({
            "year_month": int(d.strftime("%Y%m")),
            "Global_ID": [f"{country}{i:05d}" for i in range(n_firms)],
            "Barra_ID": [f"B{country}{i:04d}" for i in range(n_firms)],
            "SEDOL": None, "update_date": None,
            "msci_name": [f"{country} Corp {i}" for i in range(n_firms)],
            "msci_country": country, "gics_code": firm_sector,
            "company_market_cap_usd": mcap, "security_market_cap_usd": mcap,
            "roe_actual": roe, "roa_actual": roa,
            "roe_estimate": roe * 0.8 + 0.02 + rng.normal(0, 0.02, n_firms),
            "roa_estimate": roa * 0.8 + 0.01,
            "profitability_factor": (roe - 0.1) * 5 + rng.normal(0, 0.3, n_firms),
            "gross_margin_actual": nm + rng.uniform(0.1, 0.4, n_firms),
            "operating_margin_actual": nm * 1.4 + rng.normal(0, 0.01, n_firms),
            "net_margin_actual": nm, "asset_turnover_actual": to,
            "equity_ratio": eq, "debt_to_assets": np.clip(1 - eq - rng.uniform(0.1, 0.3, n_firms), 0, None),
            "operating_cf_to_assets": roa + rng.normal(0.03, 0.03, n_firms),
            "book_to_price_actual": rng.lognormal(np.log(0.4 if country == "US" else 0.9), 0.5, n_firms),
            "earnings_to_price_actual": rng.normal(0.05, 0.03, n_firms),
            "sales_growth_actual": rng.normal(0.05, 0.15, n_firms),
            "asset_growth_actual": rng.normal(0.05, 0.12, n_firms),
            "investment_to_assets_actual": rng.normal(0.05, 0.1, n_firms),
            "dividend_yield_actual": rng.uniform(0, 0.05, n_firms),
            "dividend_yield_estimate": rng.uniform(0, 0.05, n_firms),
            "dividend_on_equity": rng.uniform(0, 0.06, n_firms),
            "buyback_yield": rng.uniform(0, 0.04 if country == "US" else 0.015, n_firms),
            "gross_payout_yield": rng.uniform(0, 0.08, n_firms),
            "net_payout_yield": rng.uniform(-0.02, 0.07, n_firms),
        })
        rows.append(df)
    return pd.concat(rows, ignore_index=True)



def make_dummy_return_data(chars, seed=0):
    """動作確認用の架空リターン（USE_DUMMY_DATA=True のときのみ使用）。分析上の意味はない。"""
    rng = np.random.default_rng(seed + 10)
    d = chars[["year_month", "Global_ID", "Barra_ID", "roe_actual", "book_to_price_actual"]].copy()
    z = lambda s: (s - s.groupby(d["year_month"]).transform("mean")) / s.groupby(d["year_month"]).transform("std")
    mkt = d["year_month"].map(dict(zip(d["year_month"].unique(), rng.normal(0.7, 4.5, d["year_month"].nunique()))))
    r = mkt + 0.15 * z(d["roe_actual"]) + 0.15 * z(d["book_to_price_actual"]) + rng.normal(0, 7, len(d))
    out = pd.DataFrame({
        "year_month": d["year_month"], "Global_ID": d["Global_ID"], "Barra_ID": d["Barra_ID"],
        "SEDOL": None, "mkt_update_date": None,
        "return_1m_forward_local_inc_div_pct": r, "return_1m_forward_local_ex_div_pct": r - 0.15,
        "return_1m_forward_usd_inc_div_pct": r, "return_1m_forward_usd_ex_div_pct": r - 0.15,
    })
    return out[rng.random(len(out)) > 0.01]   # 1%の行を欠損させ、指数除外などを模擬


def load_country(country):
    if USE_DUMMY_DATA:
        chars = make_dummy_data(country, START_DATE, END_DATE)
        rets = make_dummy_return_data(chars)
    else:
        chars = get_profitability_data(country, START_DATE, END_DATE)
        rets = get_return_data(country, START_DATE, END_DATE)
    return chars, rets

In [ ]:
raw = {}
for c in ["US", "JP"]:
    raw[c] = load_country(c)
    print(f"{c}: 特性 {len(raw[c][0]):,} 行 / リターン {len(raw[c][1]):,} 行")

## 2. 前処理

- **ユニバース**：組成月 t に MSCI World（金融除外）に含まれ、時価総額がある銘柄
- **財務特性**（収益性・B/P・投資・還元など）：**月 t − `CHAR_LAG_MONTHS` の値** を使う（実績が判明した後に投資する想定）
- **時価総額**（サイズ分類・VWのウェイト）：組成月 t の値
- **リターン**：月 t→t+1 の将来1か月リターン（`RET_COL`, % → 小数）。**リターンが無い銘柄も除外せずに残す**（3章で再配分）
- 特性値は国×月で 1%/99% ウィンザー化（リターンは加工しない）

In [ ]:
GICS_SECTOR = {
    "10": "エネルギー", "15": "素材", "20": "資本財・サービス", "25": "一般消費財・サービス",
    "30": "生活必需品", "35": "ヘルスケア", "40": "金融", "45": "情報技術",
    "50": "コミュニケーション・サービス", "55": "公益事業", "60": "不動産",
}
LABELS = {
    "roe_actual": "ROE（実績）", "roa_actual": "ROA（実績）", "profitability_factor": "収益性ファクター",
    "book_to_price_actual": "B/P", "earnings_to_price_actual": "E/P",
    "asset_growth_actual": "総資産成長率", "investment_to_assets_actual": "投資/総資産",
    "gross_payout_yield": "総還元利回り", "net_payout_yield": "純還元利回り",
}
lab = lambda c: LABELS.get(c, c)
FUND_COLS = sorted({PROF_VAR} | {v[0] for v in ADJUSTERS.values()})   # ラグをかける財務特性


def prepare(chars, rets, country, lag=CHAR_LAG_MONTHS):
    ch = chars.copy()
    ch["Global_ID"] = ch["Global_ID"].astype(str)
    ch["date"] = pd.to_datetime(ch["year_month"].astype(int).astype(str), format="%Y%m")
    for c in FUND_COLS + ["company_market_cap_usd"]:
        ch[c] = pd.to_numeric(ch[c], errors="coerce")
        ch[c] = ch[c].mask(ch[c] <= -999999990)
    gics = pd.to_numeric(ch["gics_code"], errors="coerce").astype("Int64").astype("string")
    ch["sector_code"] = gics.str[:2]
    ch["sector"] = ch["sector_code"].map(GICS_SECTOR).fillna("不明")
    ch = ch.drop_duplicates(["date", "Global_ID"])

    # 組成月 t の情報（ユニバース・時価総額・セクター）
    base = ch.loc[~ch["sector_code"].isin(EXCLUDE_SECTOR_CODES),
                  ["date", "Global_ID", "msci_name", "sector", "company_market_cap_usd"]]
    base = base.rename(columns={"company_market_cap_usd": "mcap"})
    base = base[base["mcap"] > 0]
    # 財務特性は lag か月前の値（その時点で判明している実績）
    fund = ch[["date", "Global_ID"] + FUND_COLS].copy()
    fund["date"] = fund["date"] + pd.DateOffset(months=lag)
    d = base.merge(fund, on=["date", "Global_ID"], how="inner")

    # リターンは左結合：将来リターンが無い銘柄も残す（サバイバーシップ対策）
    rt = rets.copy()
    rt["Global_ID"] = rt["Global_ID"].astype(str)
    rt["date"] = pd.to_datetime(rt["year_month"].astype(int).astype(str), format="%Y%m")
    rt["ret"] = pd.to_numeric(rt[RET_COL], errors="coerce") / 100
    rt = rt.drop_duplicates(["date", "Global_ID"])[["date", "Global_ID", "ret"]]
    d = d.merge(rt, on=["date", "Global_ID"], how="left")
    d["country"] = country

    g = d.groupby("date")[FUND_COLS]
    lo = d[["date"]].join(g.quantile(WINSOR_LOWER), on="date")[FUND_COLS]
    hi = d[["date"]].join(g.quantile(WINSOR_UPPER), on="date")[FUND_COLS]
    d[FUND_COLS] = d[FUND_COLS].clip(lower=lo, upper=hi)
    return d.sort_values(["date", "Global_ID"]).reset_index(drop=True)


def make_return_panel(rets):
    """月 × 銘柄 の将来1か月リターン行列（指数構成期間外・欠損は NaN）。12か月保有の計算用"""
    rt = rets.copy()
    rt["Global_ID"] = rt["Global_ID"].astype(str)
    rt["date"] = pd.to_datetime(rt["year_month"].astype(int).astype(str), format="%Y%m")
    rt["ret"] = pd.to_numeric(rt[RET_COL], errors="coerce") / 100
    R = rt.drop_duplicates(["date", "Global_ID"]).pivot(index="date", columns="Global_ID", values="ret")
    return R.reindex(pd.date_range(R.index.min(), R.index.max(), freq="MS"))


data = {c: prepare(*raw[c], c) for c in ["US", "JP"]}
ret_panel = {c: make_return_panel(raw[c][1]) for c in ["US", "JP"]}
for c, d in data.items():
    print(f"{COUNTRY_LABELS[c]}: {len(d):,} 行, {d['date'].min():%Y-%m}〜{d['date'].max():%Y-%m}, "
          f"月平均 {d.groupby('date').size().mean():.0f} 銘柄, 翌月リターン欠損 {d['ret'].isna().mean():.2%}"
          f"（財務ラグ {CHAR_LAG_MONTHS}か月）")

### 先読みの診断：財務特性はいつ更新されているか

DB の月次値が「その月に判明していた実績」なのかを確かめるため、収益性（実績）の値が前月から変わった月の分布を見ます。
決算月（米国は12月期が多く2〜3月、日本は3月期が多く5〜6月）の少し後に集中していれば、決算発表に合わせて更新されていると考えられ、
`CHAR_LAG_MONTHS` はその遅れを吸収できる長さにします。また `update_date` が読めれば、月末から更新日までの日数も表示します。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8))
for c in ["US", "JP"]:
    ch = raw[c][0][["year_month", "Global_ID", PROF_VAR]].copy()
    ch[PROF_VAR] = pd.to_numeric(ch[PROF_VAR], errors="coerce")
    ch = ch.sort_values(["Global_ID", "year_month"])
    prev = ch.groupby("Global_ID")[PROF_VAR].shift()
    changed = (ch[PROF_VAR] != prev) & prev.notna() & ch[PROF_VAR].notna()
    by_m = changed.groupby(ch["year_month"].astype(int) % 100).mean()
    axes[0].plot(by_m.index, by_m.values, marker="o", color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
    print(f"{COUNTRY_LABELS[c]}: {lab(PROF_VAR)}が前月から変化した割合 {changed.mean():.1%}")
axes[0].set_xticks(range(1, 13))
axes[0].set_xlabel("月")
axes[0].set_title(f"{lab(PROF_VAR)}が前月から変化した銘柄の割合（暦月別）")
axes[0].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axes[0].legend()

try:
    plotted = False
    for c in ["US", "JP"]:
        ch = raw[c][0]
        upd = pd.to_datetime(ch["update_date"], errors="coerce")
        month_end = pd.to_datetime(ch["year_month"].astype(int).astype(str), format="%Y%m") + pd.offsets.MonthEnd(0)
        lag_days = (upd - month_end).dt.days.dropna()
        if len(lag_days):
            axes[1].hist(lag_days.clip(-400, 400), bins=60, alpha=0.5, color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
            plotted = True
            print(f"{COUNTRY_LABELS[c]}: update_date − 月末 の日数 分位点", lag_days.quantile([0.05, 0.5, 0.95]).to_dict())
    axes[1].set_title("update_date − 対象月末（日）")
    if plotted:
        axes[1].legend()
    else:
        axes[1].set_visible(False)
        print("update_date が日付として読めないため、更新日の診断は省略しました")
except Exception as e:
    axes[1].set_visible(False)
    print("update_date を日付として解釈できませんでした:", e)
plt.tight_layout()
plt.show()

In [ ]:
# 銘柄数と特性のカバレッジ
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for c in ["US", "JP"]:
    n = data[c].groupby("date").size()
    axes[0].plot(n.index, n, color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
axes[0].set_title("銘柄数（金融除外後, リターン欠損も含む）")
axes[0].legend()

cov = pd.DataFrame({COUNTRY_LABELS[c]: data[c][[PROF_VAR] + [v[0] for v in ADJUSTERS.values()]].notna().mean()
                    for c in ["US", "JP"]})
cov.index = [lab(i) for i in cov.index]
cov.plot.barh(ax=axes[1], color=[COUNTRY_COLORS["US"], COUNTRY_COLORS["JP"]])
axes[1].set_title("特性のカバレッジ（非欠損率）")
axes[1].xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

us = data["US"]
print(f"米国の 1セルあたり平均銘柄数（サイズ{N_SIZE}×調整{N_ADJ}×収益性{N_PROF}）: "
      f"{us.groupby('date').size().mean() / (N_SIZE * N_ADJ * N_PROF):.1f}")
print(f"日本の 1セルあたり平均銘柄数: {data['JP'].groupby('date').size().mean() / (N_SIZE * N_ADJ * N_PROF):.1f}")

## 3. ポートフォリオ構築の道具

- `bucket`：指定したグループ（月、月×サイズ階層 など）の中で順位を付けて N 階層に分ける（1=低, N=高）
- `port_returns`：各月・各ポートフォリオの EW / VW リターン。**翌月リターンが無い銘柄の分は、組成時のウェイトに比例して残りの銘柄に再配分**
  （＝組成時ウェイトのまま、リターンのある銘柄で加重平均するのと同じ）。最小銘柄数は組成時点の銘柄数で判定
- `sort_ls`：条件付けセル（サイズ階層など）の中で収益性階層を作り、**セル間で単純平均**した最上位−最下位（Fama-French型）
- 成績：年率平均・年率ボラ・シャープレシオ・Newey-West t値・最大ドローダウン・勝率

In [ ]:
def bucket(d, col, n, keys):
    r = d.groupby(keys)[col].rank(pct=True, method="first")
    return np.ceil(r * n).clip(1, n)


def zscore(d, col, keys):
    g = d.groupby(keys)[col]
    return (d[col] - g.transform("mean")) / g.transform("std")


def port_returns(d, group_cols, weight):
    """組成時ウェイト（EW: 等ウェイト, VW: 組成月の時価総額）。リターン欠損銘柄の分は残りの銘柄に組成時ウェイト比例で再配分"""
    x = d.dropna(subset=group_cols)
    w = x["mcap"] if weight == "VW" else pd.Series(1.0, index=x.index)
    avail = x["ret"].notna()
    x = x.assign(_w=w.where(avail, 0.0), _wr=(w * x["ret"]).where(avail, 0.0), _n=1)
    g = x.groupby(["date"] + group_cols)
    r = (g["_wr"].sum() / g["_w"].sum().replace(0, np.nan))[g["_n"].sum() >= MIN_STOCKS]
    return r.unstack(group_cols) if group_cols else r


def sort_ls(d, sort_col, cond_cols, n, weight):
    """cond_cols の各セル内で sort_col を n 階層 → セル間平均の各階層リターンと 最上位−最下位"""
    x = d.dropna(subset=[sort_col] + cond_cols).copy()
    x["pg"] = bucket(x, sort_col, n, ["date"] + cond_cols).astype(int)
    pr = port_returns(x, cond_cols + ["pg"], weight)
    legs = pr.T.groupby(level="pg").mean().T if cond_cols else pr
    legs = legs.reindex(columns=range(1, n + 1))
    return legs[n] - legs[1], legs, pr


def nw_tstat(x, lags=NW_LAGS):
    x = pd.Series(x).dropna().to_numpy()
    n = len(x)
    if n < 3:
        return np.nan
    e = x - x.mean()
    s = e @ e / n
    for l in range(1, min(lags, n - 1) + 1):
        s += 2 * (1 - l / (lags + 1)) * (e[l:] @ e[:-l]) / n
    return x.mean() / np.sqrt(s / n) if s > 0 else np.nan


def perf(s):
    s = pd.Series(s).dropna()
    if len(s) < 12:
        return pd.Series(dtype=float)
    cum = (1 + s).cumprod()
    m, v = s.mean() * 12, s.std() * np.sqrt(12)
    return pd.Series({
        "年率平均": m, "年率ボラ": v, "シャープ": m / v if v > 0 else np.nan, "t値(NW)": nw_tstat(s),
        "最大DD": (cum / cum.cummax() - 1).min(), "勝率": (s > 0).mean(), "月数": len(s),
    })


def perf_table(series_dict):
    return pd.DataFrame({k: perf(v) for k, v in series_dict.items()}).T


def ols_nw(y, X, lags=NW_LAGS):
    """OLS（定数項付き）と Newey-West 標準誤差。戻り値：係数, t値"""
    df_ = pd.concat([y, X], axis=1).dropna()
    yv = df_.iloc[:, 0].to_numpy()
    Xv = np.column_stack([np.ones(len(df_)), df_.iloc[:, 1:].to_numpy()])
    XtX_inv = np.linalg.inv(Xv.T @ Xv)
    b = XtX_inv @ Xv.T @ yv
    u = yv - Xv @ b
    Xu = Xv * u[:, None]
    S = Xu.T @ Xu
    for l in range(1, lags + 1):
        G = Xu[l:].T @ Xu[:-l]
        S += (1 - l / (lags + 1)) * (G + G.T)
    se = np.sqrt(np.diag(XtX_inv @ S @ XtX_inv))
    names = ["alpha"] + list(df_.columns[1:])
    return pd.Series(b, index=names), pd.Series(b / se, index=names)


def fmt_perf(t):
    return t.style.format({"年率平均": "{:.2%}", "年率ボラ": "{:.2%}", "シャープ": "{:.2f}", "t値(NW)": "{:.2f}",
                           "最大DD": "{:.1%}", "勝率": "{:.1%}", "月数": "{:.0f}"})

### 全ステップをまとめて計算する関数

調整変数ごとに「収益性・時価総額・リターン・調整変数がそろう銘柄」を共通サンプルとし、
その上でステップ 0 / 1 / 2 / 2b / 2c と調整変数単体のロングショートを EW・VW で計算します。

In [ ]:
METHODS = ["0 無条件", "1 サイズ", "2 サイズ→調整（2段ソート）", "2b 合成スコア", "2c 残差"]


def make_sample(d_all, adj_name):
    """共通サンプル（収益性・調整変数・時価総額がそろう銘柄。リターンの有無は問わない）と各種ソート用の列"""
    col, sign = ADJUSTERS[adj_name]
    d = d_all.dropna(subset=[PROF_VAR, col, "mcap"]).copy()
    d["adj"] = sign * d[col]
    d["size_b"] = bucket(d, "mcap", N_SIZE, ["date"]).astype(int)
    d["adj_b"] = bucket(d, "adj", N_ADJ, ["date", "size_b"]).astype(int)
    keys = ["date", "size_b"]
    zp, za = zscore(d, PROF_VAR, keys), zscore(d, "adj", keys)
    d["comp"] = zp + za
    grp = [d[k] for k in keys]
    beta = (zp * za).groupby(grp).transform("mean") / (za ** 2).groupby(grp).transform("mean")
    d["resid"] = zp - beta * za
    return d


SORT_SPECS = {   # 手法 → (並べる列, 条件付けセル)
    "0 無条件": (PROF_VAR, []),
    "1 サイズ": (PROF_VAR, ["size_b"]),
    "2 サイズ→調整（2段ソート）": (PROF_VAR, ["size_b", "adj_b"]),
    "2b 合成スコア": ("comp", ["size_b"]),
    "2c 残差": ("resid", ["size_b"]),
    "調整変数単体": ("adj", ["size_b"]),
}


def run_steps(d_all, adj_name, weight):
    d = make_sample(d_all, adj_name)
    out = {}
    out["0 無条件"] = sort_ls(d, PROF_VAR, [], N_PROF, weight)
    out["1 サイズ"] = sort_ls(d, PROF_VAR, ["size_b"], N_PROF, weight)
    out["2 サイズ→調整（2段ソート）"] = sort_ls(d, PROF_VAR, ["size_b", "adj_b"], N_PROF, weight)
    out["2b 合成スコア"] = sort_ls(d, "comp", ["size_b"], N_PROF, weight)
    out["2c 残差"] = sort_ls(d, "resid", ["size_b"], N_PROF, weight)
    out["調整変数単体"] = sort_ls(d, "adj", ["size_b"], N_PROF, weight)
    bench = port_returns(d.assign(all=1), ["all"], weight)[1]
    return {"ls": {k: v[0] for k, v in out.items()}, "legs": {k: v[1] for k, v in out.items()},
            "pr": {k: v[2] for k, v in out.items()}, "bench": bench, "sample": d}


def run_country(c):
    return {(a, w): run_steps(data[c], a, w) for a in ADJUSTERS for w in ["EW", "VW"]}


res = {"US": run_country("US")}
print("米国の計算完了")

## 4. ステップ0：無条件の収益性ポートフォリオ（米国）

まず収益性だけで並べたときに、リターンが単調に並ぶかを確認します（共通サンプルは主調整変数 `MAIN_ADJ` のもの）。

In [ ]:
base = res["US"][(MAIN_ADJ, "EW")]["sample"]
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
qret = {}
for i, w in enumerate(["EW", "VW"]):
    _, legs, _ = sort_ls(base, PROF_VAR, [], N_PROF_UNCOND, w)
    qret[w] = legs.mean() * 12
    axes[0].bar(np.arange(1, N_PROF_UNCOND + 1) + (i - 0.5) * 0.38, qret[w], 0.38,
                color=["#7f8c8d", "#2c3e50"][i], label=WEIGHT_LABELS[w])
    ls = legs[N_PROF_UNCOND] - legs[1]
    axes[1].plot(ls.index, (1 + ls).cumprod(), color=["#7f8c8d", "#2c3e50"][i], label=WEIGHT_LABELS[w])
axes[0].set_xticks(range(1, N_PROF_UNCOND + 1))
axes[0].set_xticklabels([f"Q{i}" for i in range(1, N_PROF_UNCOND + 1)])
axes[0].set_title(f"米国：{lab(PROF_VAR)}分位別の年率リターン")
axes[0].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
axes[0].legend()
axes[1].set_yscale("log")
axes[1].set_title(f"Q{N_PROF_UNCOND}−Q1 の累積リターン")
axes[1].legend()

# 収益性と調整変数の横断面の順位相関：負なら「高収益＝割高」で、バリュー調整が効きやすい構造
for c in ["US"]:
    for a, ls_ in zip(["B/P", "低投資（総資産成長率）", "総還元利回り"], ["-", "--", ":"]):
        col, sign = ADJUSTERS[a]
        x = data[c].dropna(subset=[PROF_VAR, col])
        rc = (x.groupby("date")[[PROF_VAR, col]].rank().groupby(x["date"]).corr()
              .xs(PROF_VAR, level=1)[col] * sign)
        axes[2].plot(rc.index, rc.rolling(12, min_periods=6).mean(), color=COUNTRY_COLORS[c], ls=ls_,
                     label=f"{a}（平均 {rc.mean():+.2f}）")
axes[2].axhline(0, color="black", lw=0.8)
axes[2].set_title(f"米国：{lab(PROF_VAR)}と調整変数の順位相関（12M平均）")
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. ステップ1：サイズで条件付け（米国）

サイズ階層（1=小型 … 大型）ごとに収益性ロングショートの成績を見ます。**どのサイズで収益性が効いているか** が分かります。

In [ ]:
def by_cell(pr, cond_level, n=N_PROF):
    """pr（列: cond..., pg）から cond_level ごとの 最上位−最下位（他の条件はセル間平均）"""
    hi = pr.xs(n, axis=1, level="pg").T.groupby(level=cond_level).mean().T
    lo = pr.xs(1, axis=1, level="pg").T.groupby(level=cond_level).mean().T
    return hi - lo


rows = {}
for w in ["EW", "VW"]:
    r = res["US"][(MAIN_ADJ, w)]
    s1 = by_cell(r["pr"]["1 サイズ"], "size_b")
    for b in s1.columns:
        rows[(WEIGHT_LABELS[w], f"サイズ{int(b)}")] = s1[b]
    rows[(WEIGHT_LABELS[w], "平均（サイズ中立）")] = r["ls"]["1 サイズ"]
    rows[(WEIGHT_LABELS[w], "参考：無条件")] = r["ls"]["0 無条件"]
size_tbl = perf_table(rows)
fmt_perf(size_tbl)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for ax, w in zip(axes, ["EW", "VW"]):
    pr = res["US"][(MAIN_ADJ, w)]["pr"]["1 サイズ"]
    m = pr.mean().unstack("pg") * 12
    x = np.arange(len(m))
    for j, p in enumerate(m.columns):
        ax.bar(x + (j - (N_PROF - 1) / 2) * 0.25, m[p], 0.25, label=f"収益性{int(p)}",
               color=plt.get_cmap("Greens")(0.3 + 0.6 * j / max(N_PROF - 1, 1)))
    ax.set_xticks(x)
    ax.set_xticklabels([f"サイズ{int(b)}" for b in m.index])
    ax.set_title(f"米国（{WEIGHT_LABELS[w]}）：サイズ階層×収益性階層の年率リターン")
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. ステップ2：さらにバリュー（主調整変数）で調整（米国）

### 6-1. ステップごとの成績と改善幅

In [ ]:
def step_table(c, adj, w):
    r = res[c][(adj, w)]
    t = perf_table({m: r["ls"][m] for m in METHODS + ["調整変数単体"]})
    t["Δシャープ（vs 1 サイズ）"] = t["シャープ"] - t.loc["1 サイズ", "シャープ"]
    return t


for w in ["EW", "VW"]:
    print(f"■ 米国・{WEIGHT_LABELS[w]}・調整変数={MAIN_ADJ}")
    display(fmt_perf(step_table("US", MAIN_ADJ, w)).format({"Δシャープ（vs 1 サイズ）": "{:+.2f}"}, subset=["Δシャープ（vs 1 サイズ）"]))

In [ ]:
STEP_COLORS = {"0 無条件": "#bdc3c7", "1 サイズ": "#7f8c8d", "2 サイズ→調整（2段ソート）": "#e67e22",
               "2b 合成スコア": "#27ae60", "2c 残差": "#8e44ad", "調整変数単体": "#3498db"}

fig, axes = plt.subplots(2, 2, figsize=(16, 8.5))
for j, w in enumerate(["EW", "VW"]):
    r = res["US"][(MAIN_ADJ, w)]
    for m in METHODS:
        s = r["ls"][m].dropna()
        axes[0, j].plot(s.index, (1 + s).cumprod(), color=STEP_COLORS[m], lw=2 if m.startswith("2 ") else 1.2, label=m)
    axes[0, j].set_yscale("log")
    axes[0, j].set_title(f"米国（{WEIGHT_LABELS[w]}）：収益性ロングショートの累積リターン")
    axes[0, j].legend(fontsize=8)

    roll = {m: r["ls"][m].rolling(ROLL_MONTHS).mean() / r["ls"][m].rolling(ROLL_MONTHS).std() * np.sqrt(12)
            for m in ["1 サイズ", "2 サイズ→調整（2段ソート）"]}
    for m, s in roll.items():
        axes[1, j].plot(s.index, s, color=STEP_COLORS[m], label=m)
    axes[1, j].fill_between(roll["1 サイズ"].index, roll["1 サイズ"], roll["2 サイズ→調整（2段ソート）"],
                            where=roll["2 サイズ→調整（2段ソート）"] > roll["1 サイズ"], color="#e67e22", alpha=0.15)
    axes[1, j].axhline(0, color="black", lw=0.8)
    axes[1, j].set_title(f"{ROLL_MONTHS}か月ローリング・シャープ（{WEIGHT_LABELS[w]}）：網掛け＝調整で改善した期間")
    axes[1, j].legend(fontsize=8)
plt.tight_layout()
plt.show()

### 6-2. どこで改善しているか（1）：サイズ階層別・バリュー階層別

2段ソートのセル（サイズ × 調整変数）ごとに収益性ロングショートの年率リターンを見ます。
「割安な銘柄群の中では収益性が効く」「割高な銘柄群の中では効かない」などが分かります。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, w in zip(axes, ["EW", "VW"]):
    pr = res["US"][(MAIN_ADJ, w)]["pr"]["2 サイズ→調整（2段ソート）"]
    cell = (pr.xs(N_PROF, axis=1, level="pg") - pr.xs(1, axis=1, level="pg")).mean().unstack("adj_b") * 12
    im = ax.imshow(cell.values, cmap="RdBu_r", vmin=-0.08, vmax=0.08, aspect="auto")
    for i in range(cell.shape[0]):
        for j in range(cell.shape[1]):
            ax.text(j, i, f"{cell.values[i, j]:.1%}", ha="center", va="center", fontsize=10)
    ax.set_xticks(range(cell.shape[1])); ax.set_xticklabels([f"{MAIN_ADJ}{int(b)}" for b in cell.columns])
    ax.set_yticks(range(cell.shape[0])); ax.set_yticklabels([f"サイズ{int(b)}" for b in cell.index])
    ax.set_title(f"米国（{WEIGHT_LABELS[w]}）：セル内の収益性ロングショート年率\n（{MAIN_ADJ}は 1=不利 → {N_ADJ}=有利）")
    ax.grid(False)
    plt.colorbar(im, ax=ax, shrink=0.8, format=plt.matplotlib.ticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

In [ ]:
# サイズ階層ごとに「サイズのみ」と「サイズ→調整」を同じ物差しで比較
rows = {}
for w in ["EW", "VW"]:
    r = res["US"][(MAIN_ADJ, w)]
    s1 = by_cell(r["pr"]["1 サイズ"], "size_b")
    s2 = by_cell(r["pr"]["2 サイズ→調整（2段ソート）"], "size_b")
    for b in s1.columns:
        p1, p2 = perf(s1[b]), perf(s2[b])
        rows[(WEIGHT_LABELS[w], f"サイズ{int(b)}")] = {
            "年率（サイズのみ）": p1["年率平均"], "年率（+調整）": p2["年率平均"],
            "シャープ（サイズのみ）": p1["シャープ"], "シャープ（+調整）": p2["シャープ"],
            "Δシャープ": p2["シャープ"] - p1["シャープ"]}
size_imp = pd.DataFrame(rows).T
size_imp.style.format({c: "{:.2%}" for c in size_imp.columns if "年率" in c} |
                      {c: "{:+.2f}" if c.startswith("Δ") else "{:.2f}" for c in size_imp.columns if "シャープ" in c})

### 6-3. どこで改善しているか（2）：ロング側とショート側

収益性の上位ポートフォリオ（ロング）と下位（ショート）それぞれの、ユニバース平均（同じウェイト方式）に対する超過リターンです。
調整によって **「良いロングが選べるようになった」のか「悪いショートが避けられるようになった」のか** を区別します。

In [ ]:
rows = {}
for w in ["EW", "VW"]:
    r = res["US"][(MAIN_ADJ, w)]
    for m in METHODS:
        legs = r["legs"][m]
        ex_long = (legs[N_PROF] - r["bench"]).dropna()
        ex_short = (r["bench"] - legs[1]).dropna()
        rows[(WEIGHT_LABELS[w], m)] = {
            "ロング超過（年率）": ex_long.mean() * 12, "ロングt": nw_tstat(ex_long),
            "ショート寄与（年率）": ex_short.mean() * 12, "ショートt": nw_tstat(ex_short)}
legtbl = pd.DataFrame(rows).T

fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=True)
for ax, w in zip(axes, ["EW", "VW"]):
    t = legtbl.loc[WEIGHT_LABELS[w]]
    x = np.arange(len(t))
    ax.bar(x - 0.2, t["ロング超過（年率）"], 0.4, color="#27ae60", label="ロング側（上位−ユニバース）")
    ax.bar(x + 0.2, t["ショート寄与（年率）"], 0.4, color="#c0392b", label="ショート側（ユニバース−下位）")
    ax.set_xticks(x)
    ax.set_xticklabels(t.index, rotation=20, ha="right", fontsize=8)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"米国（{WEIGHT_LABELS[w]}）：ロング/ショート別の寄与")
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
legtbl.style.format({"ロング超過（年率）": "{:.2%}", "ショート寄与（年率）": "{:.2%}", "ロングt": "{:.2f}", "ショートt": "{:.2f}"})

### 6-4. 改善は「バリューを足しただけ」か：スパニング回帰

調整後の収益性ロングショートを、**サイズ中立の収益性ロングショート** と **サイズ中立の調整変数ロングショート** に回帰します。
アルファが有意なら、両者を単に組み合わせた以上の改善（収益性とバリューの相互作用）があることを意味します。

In [ ]:
rows = {}
for w in ["EW", "VW"]:
    r = res["US"][(MAIN_ADJ, w)]
    X = pd.concat({"収益性(サイズ中立)": r["ls"]["1 サイズ"], f"{MAIN_ADJ}(サイズ中立)": r["ls"]["調整変数単体"]}, axis=1)
    for m in ["2 サイズ→調整（2段ソート）", "2b 合成スコア", "2c 残差"]:
        b, t = ols_nw(r["ls"][m].rename("y"), X)
        rows[(WEIGHT_LABELS[w], m)] = {"α（年率）": b["alpha"] * 12, "α t値": t["alpha"],
                                     "β 収益性": b.iloc[1], "β 調整変数": b.iloc[2], "t 調整変数": t.iloc[2]}
span = pd.DataFrame(rows).T
span.style.format({"α（年率）": "{:.2%}", "α t値": "{:.2f}", "β 収益性": "{:.2f}", "β 調整変数": "{:.2f}", "t 調整変数": "{:.2f}"})

### 6-5. 期間別

In [ ]:
rows = {}
for w in ["EW", "VW"]:
    r = res["US"][(MAIN_ADJ, w)]
    for pname, (y0, y1) in PERIODS.items():
        for m in ["1 サイズ", "2 サイズ→調整（2段ソート）", "2b 合成スコア"]:
            s = r["ls"][m]
            s = s[(s.index.year >= y0) & (s.index.year <= y1)]
            rows[(WEIGHT_LABELS[w], pname, m)] = perf(s)[["年率平均", "シャープ", "t値(NW)"]]
per_tbl = pd.DataFrame(rows).T
per_tbl.style.format({"年率平均": "{:.2%}", "シャープ": "{:.2f}", "t値(NW)": "{:.2f}"})

## 7. 調整変数の比較：バリューか、投資か、株主還元か（米国）

全ての調整変数・手法について、ステップ1（サイズのみ, 同じサンプル）からのシャープレシオ改善幅を比べます。

In [ ]:
def sharpe_grid(c, w, delta=True):
    g = {}
    for a in ADJUSTERS:
        r = res[c][(a, w)]
        sr = {m: perf(r["ls"][m]).get("シャープ", np.nan) for m in METHODS + ["調整変数単体"]}
        base_ = sr["1 サイズ"]
        g[a] = {m: (v - base_ if delta and m not in ("0 無条件", "1 サイズ", "調整変数単体") else v) for m, v in sr.items()}
    return pd.DataFrame(g).T


def show_grid(c):
    fig, axes = plt.subplots(1, 2, figsize=(16, 4.2))
    for ax, w in zip(axes, ["EW", "VW"]):
        gtab = sharpe_grid(c, w)
        cols = ["1 サイズ", "2 サイズ→調整（2段ソート）", "2b 合成スコア", "2c 残差", "調整変数単体"]
        gtab = gtab[cols]
        colored = gtab.values.astype(float).copy()
        colored[:, [0, 4]] = np.nan                        # 水準（シャープ）の列は灰色、改善幅（Δ）の列だけ色付け
        cm = plt.get_cmap("RdBu_r").copy()
        cm.set_bad("#eeeeee")
        ax.imshow(np.ma.masked_invalid(colored), cmap=cm, vmin=-0.6, vmax=0.6, aspect="auto")
        for i in range(gtab.shape[0]):
            for j in range(gtab.shape[1]):
                v = gtab.values[i, j]
                dark = j not in (0, 4) and abs(v) > 0.4
                ax.text(j, i, ("" if j in (0, 4) else "Δ") + f"{v:+.2f}", ha="center", va="center", fontsize=9,
                        color="white" if dark else "black")
        ax.set_xticks(range(len(cols)))
        ax.set_xticklabels(["サイズのみ\n(シャープ)", "2段ソート\n(Δ)", "合成\n(Δ)", "残差\n(Δ)", "調整変数単体\n(シャープ)"], fontsize=8)
        ax.set_yticks(range(len(gtab))); ax.set_yticklabels(gtab.index)
        ax.set_title(f"{COUNTRY_LABELS[c]}（{WEIGHT_LABELS[w]}）：収益性L/Sのシャープとサイズのみからの改善幅")
        ax.grid(False)
    plt.tight_layout()
    plt.show()


show_grid("US")

In [ ]:
# 累積リターン比較（2段ソート版）：調整変数ごと
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
cmap = plt.get_cmap("tab10")
for ax, w in zip(axes, ["EW", "VW"]):
    s = res["US"][(MAIN_ADJ, w)]["ls"]["1 サイズ"].dropna()
    ax.plot(s.index, (1 + s).cumprod(), color="black", lw=2, label=f"サイズのみ（{MAIN_ADJ}サンプル）")
    for i, a in enumerate(ADJUSTERS):
        s = res["US"][(a, w)]["ls"]["2 サイズ→調整（2段ソート）"].dropna()
        ax.plot(s.index, (1 + s).cumprod(), color=cmap(i), lw=1.2, label=a)
    ax.set_yscale("log")
    ax.set_title(f"米国（{WEIGHT_LABELS[w]}）：サイズ→調整変数→収益性 の累積リターン")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 8. 参考：日本との比較

日本（MSCI World 日本構成銘柄）は銘柄数が少なく、2段ソートの1セルあたり銘柄数が小さいため **参考値** として扱ってください。
同じパラメータ・同じ手順で計算しています。

In [ ]:
res["JP"] = run_country("JP")

rows = {}
for c in ["US", "JP"]:
    for w in ["EW", "VW"]:
        t = step_table(c, MAIN_ADJ, w)
        for m in METHODS + ["調整変数単体"]:
            rows[(COUNTRY_LABELS[c], WEIGHT_LABELS[w], m)] = t.loc[m, ["年率平均", "シャープ", "t値(NW)", "Δシャープ（vs 1 サイズ）"]]
cmp_tbl = pd.DataFrame(rows).T
cmp_tbl.style.format({"年率平均": "{:.2%}", "シャープ": "{:.2f}", "t値(NW)": "{:.2f}", "Δシャープ（vs 1 サイズ）": "{:+.2f}"})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.3), sharey=True)
show_m = ["0 無条件", "1 サイズ", "2 サイズ→調整（2段ソート）", "2b 合成スコア", "2c 残差"]
for ax, w in zip(axes, ["EW", "VW"]):
    x = np.arange(len(show_m))
    for i, c in enumerate(["US", "JP"]):
        sr = [perf(res[c][(MAIN_ADJ, w)]["ls"][m])["シャープ"] for m in show_m]
        ax.bar(x + (i - 0.5) * 0.38, sr, 0.38, color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
    ax.set_xticks(x)
    ax.set_xticklabels(show_m, rotation=15, ha="right", fontsize=8)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"収益性L/Sのシャープ：ステップごとの日米比較（{WEIGHT_LABELS[w]}, 調整={MAIN_ADJ}）")
    ax.legend()
plt.tight_layout()
plt.show()

show_grid("JP")

In [ ]:
# 収益性と調整変数の順位相関：日米比較（相関が負で大きいほど、調整しないと収益性の効果がバリューで打ち消される）
rows = {}
for c in ["US", "JP"]:
    for a, (col, sign) in ADJUSTERS.items():
        x = data[c].dropna(subset=[PROF_VAR, col])
        rc = x.groupby("date")[[PROF_VAR, col]].rank().groupby(x["date"]).corr().xs(PROF_VAR, level=1)[col] * sign
        rows[(a, COUNTRY_LABELS[c])] = rc.mean()
corr_tbl = pd.Series(rows).unstack()
ax = corr_tbl.plot.barh(color=[COUNTRY_COLORS["JP"], COUNTRY_COLORS["US"]], figsize=(9, 4))
ax.axvline(0, color="black", lw=0.8)
ax.set_title(f"{lab(PROF_VAR)} と調整変数（良い向き）の平均順位相関")
plt.tight_layout()
plt.show()
corr_tbl

## 8-1. 収益性ファクターの分解：日米差はバリューの影響で説明できるか

**作り方を日米でそろえた収益性ファクター**（サイズで条件付けした収益性ロングショート、B/P で調整しない）を、
同じ作り方の **B/P ファクター**（サイズで条件付けした B/P ロングショート）などに時系列回帰します。

$$\text{収益性} = \alpha + \beta_{B/P}\,\text{B/P} + \beta_{投資}\,\text{低投資} + \beta_{市場}\,\text{市場} + \varepsilon$$

- **β_B/P**：収益性ファクターに紛れ込んでいる「バリューの逆張り」の大きさ（高ROEは割高なので、日米ともマイナスのはず）
- **β × B/Pファクターの平均**：その逆張りで得した／損した分（米国はバリューが負けた期間が長いので得、日本は損のはず）
- **α**：バリュー・投資・市場の影響を除いた **純粋な収益性の効果**

**読み方**：α が日米で同程度なら、日米差は「バリューの成績の違い」で説明できる。α にも差が残れば、
その差が持続性・ミスプライシング（3本目のノートブック）で説明すべき部分。

In [ ]:
FAC_LABELS = {"VALUE": "B/P", "INV": "低投資（総資産成長率）", "MKT": "市場"}


def factor_frame(c, w):
    rb = res[c][("B/P", w)]
    ri = res[c][("低投資（総資産成長率）", w)]
    return pd.DataFrame({
        "PROF": rb["ls"]["1 サイズ"],           # 作り方をそろえた収益性ファクター（B/P 調整なし）
        "VALUE": rb["ls"]["調整変数単体"],      # サイズで条件付けした B/P ロングショート
        "INV": ri["ls"]["調整変数単体"],        # サイズで条件付けした 低投資 ロングショート
        "MKT": rb["bench"],                     # ユニバース平均（無リスク金利は差し引いていない）
    }).dropna()


def decompose(f, xcols):
    b, t = ols_nw(f["PROF"], f[xcols])
    out = {"収益性 平均(年率)": f["PROF"].mean() * 12, "α(年率)": b["alpha"] * 12, "α t値": t["alpha"]}
    for x in xcols:
        out[f"β {FAC_LABELS[x]}"] = b[x]
        out[f"t {FAC_LABELS[x]}"] = t[x]
        out[f"寄与 {FAC_LABELS[x]}(年率)"] = b[x] * f[x].mean() * 12
        out[f"{FAC_LABELS[x]} 平均(年率)"] = f[x].mean() * 12
    return pd.Series(out)


rows = {}
for c in ["US", "JP"]:
    for w in ["EW", "VW"]:
        f = factor_frame(c, w)
        rows[(COUNTRY_LABELS[c], WEIGHT_LABELS[w], "(1) B/Pのみ")] = decompose(f, ["VALUE"])
        rows[(COUNTRY_LABELS[c], WEIGHT_LABELS[w], "(2) B/P＋投資＋市場")] = decompose(f, ["VALUE", "INV", "MKT"])
decomp = pd.DataFrame(rows).T
fmt = {c: ("{:.2%}" if ("年率" in c) else "{:.2f}") for c in decomp.columns}
decomp.style.format(fmt, na_rep="")

In [ ]:
# 分解のグラフ：収益性ファクターの平均リターン ＝ α ＋ バリューの寄与 ＋ 投資の寄与 ＋ 市場の寄与
fig, axes = plt.subplots(1, 2, figsize=(15, 4.3), sharey=True)
parts = [("α(年率)", "純粋な収益性 α", "#2c3e50"), ("寄与 B/P(年率)", "B/P（バリュー逆張り）の寄与", "#e67e22"),
         ("寄与 低投資（総資産成長率）(年率)", "投資の寄与", "#27ae60"), ("寄与 市場(年率)", "市場の寄与", "#bdc3c7")]
for ax, w in zip(axes, ["EW", "VW"]):
    labels_ = [COUNTRY_LABELS[c] for c in ["US", "JP"]]
    x = np.arange(len(labels_))
    pos, neg = np.zeros(len(x)), np.zeros(len(x))
    for col, name, colr in parts:
        v = np.array([decomp.loc[(l, WEIGHT_LABELS[w], "(2) B/P＋投資＋市場"), col] for l in labels_], dtype=float)
        bottom = np.where(v >= 0, pos, neg)
        ax.bar(x, v, 0.5, bottom=bottom, color=colr, label=name)
        pos, neg = np.where(v >= 0, pos + v, pos), np.where(v < 0, neg + v, neg)
    tot = [decomp.loc[(l, WEIGHT_LABELS[w], "(2) B/P＋投資＋市場"), "収益性 平均(年率)"] for l in labels_]
    ax.scatter(x, tot, color="black", zorder=5, marker="D", label="収益性ファクターの実際の平均")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels_)
    ax.set_title(f"収益性ファクターの分解（{WEIGHT_LABELS[w]}, 年率）")
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 期間別：バリューの成績が期間によって違うなら、収益性ファクターの日米差も期間で変わるはず
rows = {}
for c in ["US", "JP"]:
    for w in ["EW", "VW"]:
        f_all = factor_frame(c, w)
        for pname, (y0, y1) in PERIODS.items():
            f = f_all[(f_all.index.year >= y0) & (f_all.index.year <= y1)]
            if len(f) < 24:
                continue
            d_ = decompose(f, ["VALUE"])
            rows[(pname, WEIGHT_LABELS[w], COUNTRY_LABELS[c])] = d_[["収益性 平均(年率)", "B/P 平均(年率)", "β B/P",
                                                                    "寄与 B/P(年率)", "α(年率)", "α t値"]]
per_decomp = pd.DataFrame(rows).T
print("【送付②】収益性ファクターの分解・期間別")
per_decomp.style.format({"収益性 平均(年率)": "{:.2%}", "B/P 平均(年率)": "{:.2%}", "β B/P": "{:.2f}",
                         "寄与 B/P(年率)": "{:.2%}", "α(年率)": "{:.2%}", "α t値": "{:.2f}"})

In [ ]:
# 36か月ローリングの β_B/P と、バリューの成績の推移
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for c in ["US", "JP"]:
    f = factor_frame(c, "EW")
    cov = f["PROF"].rolling(ROLL_MONTHS).cov(f["VALUE"])
    beta = cov / f["VALUE"].rolling(ROLL_MONTHS).var()
    axes[0].plot(beta.index, beta, color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
    axes[1].plot(f.index, (1 + f["VALUE"]).cumprod(), color=COUNTRY_COLORS[c], label=COUNTRY_LABELS[c])
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_title(f"収益性ファクターの β_B/P（{ROLL_MONTHS}か月ローリング, 等ウェイト）")
axes[0].legend()
axes[1].set_yscale("log")
axes[1].set_title("B/P ファクター（サイズ条件付き, 等ウェイト）の累積リターン")
axes[1].legend()
plt.tight_layout()
plt.show()

print("【送付①】収益性ファクターの分解（年率）")
for c in ["US", "JP"]:
    for w in ["EW", "VW"]:
        r = decomp.loc[(COUNTRY_LABELS[c], WEIGHT_LABELS[w], "(2) B/P＋投資＋市場")]
        print(f"{COUNTRY_LABELS[c]}・{WEIGHT_LABELS[w]}: 収益性 {r['収益性 平均(年率)']:+.2%} ＝ α {r['α(年率)']:+.2%}（t={r['α t値']:.1f}）"
              f" ＋ B/P寄与 {r['寄与 B/P(年率)']:+.2%}（β={r['β B/P']:.2f}, B/P平均 {r['B/P 平均(年率)']:+.2%}）"
              f" ＋ 投資寄与 {r['寄与 低投資（総資産成長率）(年率)']:+.2%} ＋ 市場寄与 {r['寄与 市場(年率)']:+.2%}")

## 8-2. 先読みラグの感応度（米国）

財務特性のラグ（`LAG_SENSITIVITY` か月）を変えても結論が変わらないかを確認します。ラグ0は先読みの可能性がある参考値です。

In [ ]:
rows = {}
for lag in LAG_SENSITIVITY:
    d_l = prepare(*raw["US"], "US", lag=lag)
    for w in ["EW", "VW"]:
        r = run_steps(d_l, MAIN_ADJ, w)
        sr = {m: perf(r["ls"][m])["シャープ"] for m in METHODS}
        rows[(f"ラグ{lag}か月", WEIGHT_LABELS[w])] = {**sr, "Δ（2段ソート−サイズ）": sr["2 サイズ→調整（2段ソート）"] - sr["1 サイズ"]}
lag_tbl = pd.DataFrame(rows).T
lag_tbl.style.format("{:.2f}")

## 9. 追加分析：12か月保有（年次ホライズン）

収益性は長めのホライズンで効くと考えられるため、月次リバランスではなく **12か月保有** で評価します。

- **組成**：毎月、1か月版と同じ方法（同じラグ・同じソート）でポートフォリオを作る
- **保有**：組成時ウェイト（EW: 等ウェイト / VW: 組成月の時価総額）で買い、12か月 **バイ・アンド・ホールド**（途中リバランスなし）
- **サバイバーシップ対策**：保有中にリターンが得られなくなった銘柄（指数除外・上場廃止など）は、その時点の評価額を
  **同じポートフォリオの残りの銘柄に組成時ウェイト比例で再配分**し、以降は保有しない
- **集計**：
  - 12か月リターン：組成月ごとのロングショート12か月リターン（重複するため t値は Newey-West ラグ12）
  - 月次換算：毎月組成した12本のポートフォリオを並行保有する戦略（Jegadeesh-Titman型）の月次リターン → シャープを1か月版と比較
  - 経過月数別の累積リターン：収益性プレミアムが保有期間中のいつ実現するか

In [ ]:
def holding_returns(d, sort_col, cond_cols, n, weight, R, hold=HOLD_MONTHS):
    '''毎月組成・hold か月バイ・アンド・ホールド。欠損銘柄の評価額は組成時ウェイト比例で残りに再配分。
    戻り値：(組成月, 経過月k, セル…, pg) ごとの月次リターン（長い形式）'''
    x = d.dropna(subset=[sort_col] + cond_cols).copy()
    x["pg"] = bucket(x, sort_col, n, ["date"] + cond_cols).astype(int)
    cells = cond_cols + ["pg"]
    x["pid"] = x.groupby(cells).ngroup()
    keymap = x.drop_duplicates("pid").set_index("pid")[cells].sort_index()
    P = int(x["pid"].max()) + 1
    Rv = R.to_numpy()
    pos = {t: i for i, t in enumerate(R.index)}
    gid_idx = pd.Index(R.columns)
    recs = []
    for t0, g in x.groupby("date"):
        i0 = pos.get(t0)
        if i0 is None:
            continue
        cols = gid_idx.get_indexer(g["Global_ID"])
        pid = g["pid"].to_numpy()
        w = g["mcap"].to_numpy(float) if weight == "VW" else np.ones(len(g))
        w0 = w / np.bincount(pid, w, P)[pid]                  # 組成時ウェイト（ポートフォリオ内で合計1）
        cnt = np.bincount(pid, minlength=P)
        ok = np.where(cnt >= MIN_STOCKS)[0]
        V = w0.copy()
        alive = np.ones(len(g), bool)
        for k in range(hold):
            if i0 + k >= len(Rv):
                break
            r = np.where(cols >= 0, Rv[i0 + k, np.clip(cols, 0, None)], np.nan)
            dead = alive & np.isnan(r)
            if dead.any():                                     # 評価額を残りの銘柄に組成時ウェイト比例で再配分
                amt = np.bincount(pid[dead], V[dead], P)
                alive &= ~np.isnan(r)
                recv = np.where(alive, w0, 0.0)
                den = np.bincount(pid, recv, P)
                share = np.divide(amt, den, out=np.zeros(P), where=den > 0)
                V = np.where(alive, V + recv * share[pid], 0.0)
            before = np.bincount(pid, V, P)
            V = np.where(alive, V * (1 + np.nan_to_num(r)), 0.0)
            after = np.bincount(pid, V, P)
            ret = np.divide(after, before, out=np.full(P, np.nan), where=before > 0) - 1
            recs.append((np.full(len(ok), t0), np.full(len(ok), k + 1), ok, ret[ok]))
    out = pd.DataFrame({"t0": np.concatenate([a[0] for a in recs]), "k": np.concatenate([a[1] for a in recs]),
                        "pid": np.concatenate([a[2] for a in recs]), "ret": np.concatenate([a[3] for a in recs])})
    return out.join(keymap, on="pid")


def summarize_holding(h, cond_cols, n, hold=HOLD_MONTHS):
    # (1) 12か月バイ・アンド・ホールド：セルごとに複利 → セル間平均 → 最上位−最下位
    cell12 = h.groupby(["t0"] + cond_cols + ["pg"])["ret"].agg(
        r12=lambda s: np.expm1(np.log1p(s).sum()), m="count")
    cell12 = cell12[cell12["m"] == hold]["r12"]
    legs12 = cell12.groupby(["t0", "pg"]).mean().unstack("pg").reindex(columns=range(1, n + 1))
    ls12 = legs12[n] - legs12[1]
    # (2) 月次換算：経過月ごとのセル間平均 → 暦月（実現月）で12コホート平均
    legs_k = h.groupby(["t0", "k", "pg"])["ret"].mean().unstack("pg").reindex(columns=range(1, n + 1))
    t0 = legs_k.index.get_level_values("t0")
    k = legs_k.index.get_level_values("k")
    cal = pd.DatetimeIndex([a + pd.DateOffset(months=int(b) - 1) for a, b in zip(t0, k)])
    legs_m = legs_k.groupby(cal).mean()
    n_coh = legs_k.groupby(cal).size()
    legs_m = legs_m[n_coh >= hold]                             # 12コホートがそろう月のみ
    # (3) 経過月別のロングショート平均（イベントタイム）
    ls_k = (legs_k[n] - legs_k[1]).groupby(level="k").mean()
    return {"ls12": ls12, "legs12": legs12, "ls_m": legs_m[n] - legs_m[1], "ls_k": ls_k}


def run_holding(c, adj_name, weight, methods=METHODS):
    d = make_sample(data[c], adj_name)
    out = {}
    for m in methods:
        col, cond = SORT_SPECS[m]
        out[m] = summarize_holding(holding_returns(d, col, cond, N_PROF, weight, ret_panel[c]), cond, N_PROF)
    return out


hold_res = {(c, w): run_holding(c, MAIN_ADJ, w) for c in ["US", "JP"] for w in ["EW", "VW"]}
print("12か月保有の計算完了")

In [ ]:
# 保有中にリターンが途切れて再配分が起きた銘柄の割合（サバイバーシップ対策の影響の大きさ）
for c in ["US", "JP"]:
    d = make_sample(data[c], MAIN_ADJ)
    R = ret_panel[c]
    cols = pd.Index(R.columns)
    lost = []
    for t0, g in d.groupby("date"):
        if t0 not in R.index:
            continue
        i0 = R.index.get_loc(t0)
        sub = R.iloc[i0:i0 + HOLD_MONTHS].reindex(columns=g["Global_ID"])
        if len(sub) == HOLD_MONTHS:
            lost.append(sub.isna().any().mean())
    print(f"{COUNTRY_LABELS[c]}: 12か月の保有中にリターンが途切れる銘柄の割合（平均） {np.mean(lost):.1%}, "
          f"翌月だけで見た割合 {d['ret'].isna().mean():.2%}")

In [ ]:
def ann_stats(ls12):
    s = ls12.dropna()
    return pd.Series({"12か月L/S平均": s.mean(), "t値(NW,ラグ12)": nw_tstat(s, lags=HOLD_MONTHS),
                      "勝率": (s > 0).mean(), "組成月数": len(s)})


rows = {}
for (c, w), hr in hold_res.items():
    for m in METHODS:
        one = perf(res[c][(MAIN_ADJ, w)]["ls"][m]) if c in res else pd.Series(dtype=float)
        hm = perf(hr[m]["ls_m"])
        a = ann_stats(hr[m]["ls12"])
        rows[(COUNTRY_LABELS[c], WEIGHT_LABELS[w], m)] = {
            "1か月版 シャープ": one.get("シャープ", np.nan),
            "12か月保有 シャープ(月次換算)": hm.get("シャープ", np.nan),
            "12か月保有 年率(月次換算)": hm.get("年率平均", np.nan),
            **a.to_dict()}
hold_tbl = pd.DataFrame(rows).T
base1 = hold_tbl.xs("1 サイズ", level=2)["12か月保有 シャープ(月次換算)"]
hold_tbl["Δシャープ（vs 1 サイズ, 12か月）"] = [
    v - base1.loc[(i[0], i[1])] for i, v in hold_tbl["12か月保有 シャープ(月次換算)"].items()]
hold_tbl.style.format({"1か月版 シャープ": "{:.2f}", "12か月保有 シャープ(月次換算)": "{:.2f}",
                       "12か月保有 年率(月次換算)": "{:.2%}", "12か月L/S平均": "{:.2%}", "t値(NW,ラグ12)": "{:.2f}",
                       "勝率": "{:.1%}", "組成月数": "{:.0f}", "Δシャープ（vs 1 サイズ, 12か月）": "{:+.2f}"})

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8.5))
for j, w in enumerate(["EW", "VW"]):
    hr = hold_res[("US", w)]
    # 経過月別の累積ロングショート（イベントタイム）
    for m in METHODS:
        cum = np.cumprod(1 + hr[m]["ls_k"]) - 1
        axes[0, j].plot(cum.index, cum.values, marker="o", ms=3, color=STEP_COLORS[m],
                        lw=2 if m.startswith("2 ") else 1.2, label=m)
    axes[0, j].axhline(0, color="black", lw=0.8)
    axes[0, j].set_xticks(range(1, HOLD_MONTHS + 1))
    axes[0, j].set_xlabel("組成からの経過月")
    axes[0, j].set_title(f"米国（{WEIGHT_LABELS[w]}）：経過月別の累積ロングショート（平均）")
    axes[0, j].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    axes[0, j].legend(fontsize=8)
    # 組成月ごとの12か月リターン（12か月移動平均）
    for m in ["1 サイズ", "2 サイズ→調整（2段ソート）", "2b 合成スコア"]:
        s = hr[m]["ls12"]
        axes[1, j].plot(s.index, s.rolling(12, min_periods=6).mean(), color=STEP_COLORS[m], label=m)
    axes[1, j].axhline(0, color="black", lw=0.8)
    axes[1, j].set_title(f"米国（{WEIGHT_LABELS[w]}）：12か月保有L/Sリターン（組成月ごと, 12M平均）")
    axes[1, j].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    axes[1, j].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# 1か月版 vs 12か月保有：シャープの比較（日米）
fig, axes = plt.subplots(1, 2, figsize=(16, 4.3), sharey=True)
for ax, w in zip(axes, ["EW", "VW"]):
    x = np.arange(len(METHODS))
    for i, (c, hz) in enumerate([("US", "1か月版 シャープ"), ("US", "12か月保有 シャープ(月次換算)"),
                                 ("JP", "1か月版 シャープ"), ("JP", "12か月保有 シャープ(月次換算)")]):
        v = hold_tbl.loc[(COUNTRY_LABELS[c], WEIGHT_LABELS[w]), hz].reindex(METHODS)
        ax.bar(x + (i - 1.5) * 0.2, v, 0.2, color=COUNTRY_COLORS[c], alpha=0.45 if hz.startswith("1か月") else 1.0,
               label=f"{COUNTRY_LABELS[c]}:{'1か月' if hz.startswith('1か月') else '12か月保有'}")
    ax.set_xticks(x)
    ax.set_xticklabels(METHODS, rotation=15, ha="right", fontsize=8)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"収益性L/Sのシャープ：保有期間別（{WEIGHT_LABELS[w]}, 調整={MAIN_ADJ}）")
    ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 10. まとめ（自動抽出）

In [ ]:
def summarize(c):
    lines = [f"■ {COUNTRY_LABELS[c]}（{lab(PROF_VAR)}, サイズ{N_SIZE}×調整{N_ADJ}×収益性{N_PROF}）"]
    for w in ["EW", "VW"]:
        t = step_table(c, MAIN_ADJ, w)
        lines.append(f"  [{WEIGHT_LABELS[w]}] シャープ: 無条件 {t.loc['0 無条件', 'シャープ']:.2f} → サイズ {t.loc['1 サイズ', 'シャープ']:.2f}"
                     f" → +{MAIN_ADJ} 2段ソート {t.loc['2 サイズ→調整（2段ソート）', 'シャープ']:.2f}"
                     f" / 合成 {t.loc['2b 合成スコア', 'シャープ']:.2f} / 残差 {t.loc['2c 残差', 'シャープ']:.2f}")
        g = sharpe_grid(c, w)[["2 サイズ→調整（2段ソート）", "2b 合成スコア", "2c 残差"]]
        best = g.stack().idxmax()
        lines.append(f"      最も改善した組み合わせ: {best[0]} × {best[1]}（Δシャープ {g.stack().max():+.2f}）")
        pr = res[c][(MAIN_ADJ, w)]["pr"]["2 サイズ→調整（2段ソート）"]
        cell = (pr.xs(N_PROF, axis=1, level="pg") - pr.xs(1, axis=1, level="pg")).mean() * 12
        lines.append(f"      収益性L/Sが最も効くセル: サイズ{int(cell.idxmax()[0])}×{MAIN_ADJ}{int(cell.idxmax()[1])}（年率 {cell.max():.1%}）"
                     f" / 最も効かないセル: サイズ{int(cell.idxmin()[0])}×{MAIN_ADJ}{int(cell.idxmin()[1])}（年率 {cell.min():.1%}）")
        ht = hold_tbl.loc[(COUNTRY_LABELS[c], WEIGHT_LABELS[w])]
        lines.append(f"      12か月保有: L/S平均 サイズ {ht.loc['1 サイズ', '12か月L/S平均']:.1%}"
                     f"（t={ht.loc['1 サイズ', 't値(NW,ラグ12)']:.1f}） → +{MAIN_ADJ} 2段ソート "
                     f"{ht.loc['2 サイズ→調整（2段ソート）', '12か月L/S平均']:.1%}"
                     f"（t={ht.loc['2 サイズ→調整（2段ソート）', 't値(NW,ラグ12)']:.1f}）")
    return "\n".join(lines)


print(f"財務ラグ {CHAR_LAG_MONTHS}か月 / リターン欠損は組成時ウェイトで再配分\n")
print("■ 収益性ファクターの分解（作り方をそろえた版, B/P＋投資＋市場で回帰）")
for c in ["US", "JP"]:
    for w in ["EW", "VW"]:
        r = decomp.loc[(COUNTRY_LABELS[c], WEIGHT_LABELS[w], "(2) B/P＋投資＋市場")]
        print(f"  {COUNTRY_LABELS[c]}・{WEIGHT_LABELS[w]}: 収益性 {r['収益性 平均(年率)']:+.2%} ＝ 純粋な収益性α {r['α(年率)']:+.2%}"
              f"（t={r['α t値']:.1f}） ＋ バリュー逆張りの寄与 {r['寄与 B/P(年率)']:+.2%}（β={r['β B/P']:.2f}）")
print()
print(summarize("US"))
print()
print(summarize("JP"), "  ※参考値（銘柄数が少ない）")

### 読み方のポイント

1. **米国でも調整で改善するか**：6-1 の Δシャープ（vs サイズのみ）が EW・VW ともにプラスで、t値も伴うか
2. **どこで効くか**：6-2 のセル図で、収益性が効くのが「割安×小型」なのか「大型」なのか。日本と同じパターンか
3. **ロングかショートか**：6-3 で改善がロング側（割安な高収益を拾う）由来か、ショート側（割高な低収益を避ける）由来か
4. **単なる足し算か**：6-4 のαが有意でなければ「収益性とバリューを別々に持つ」のと大差ない
   （特に 2b 合成スコアは調整変数自体のプレミアムも取り込むため、シャープ改善が大きく出やすい。αで確認すること。2段ソート・残差は調整変数の影響を除いた『純粋な収益性』の効果）
5. **バリュー／投資／還元のどれで調整すべきか**：7章のグリッド。米国は自社株買いが大きいため、還元で調整した方が効く可能性
6. **日米の違いの原因**：8章の順位相関。日本の方が「高収益＝割高」の負の相関が強ければ、調整の効果が日本で大きいことと整合的
7. **ホライズン**：9章で12か月保有の方がシャープ・t値が高ければ、収益性は長期で効くファクター。経過月別の累積で、調整後の上乗せが
   保有期間を通じて積み上がるか（持続的）、最初の数か月で消えるか（短期的）を確認
8. **先読みの影響**：8-2章でラグを長くしても調整の効果が残るか